# SAE Analysis for MusicGen Layer 22

Analyzes the TopK Sparse Autoencoder trained on MusicGen decoder activations.

In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from collections import Counter

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cuda


In [3]:
# Paths
LAYER_IDX = 22
CHECKPOINT_PATH = Path("checkpoints") / f"sae_layer{LAYER_IDX:02d}.pt"
DATA_PATH = Path(f"/home/harinit9/orcd/pool/musicgen-activations-nokey/acts_by_layer/layer_{LAYER_IDX:02d}.npy")
METADATA_PATH = Path("/home/harinit9/orcd/pool/musicgen-data-nokey/dataset_metadata.json")

# Load checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
print(f"Loaded: d_in={ckpt['d_in']}, d_hidden={ckpt['d_hidden']}, top_k={ckpt.get('top_k')}")


Loaded: d_in=2048, d_hidden=8192, top_k=64


In [4]:
# Define TopK SAE model
class SAE(nn.Module):
    def __init__(self, d_in, d_hidden, top_k=64):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_hidden, bias=True)
        self.decoder = nn.Linear(d_hidden, d_in, bias=False)
        self.top_k = top_k

    def forward(self, x):
        z_pre = self.encoder(x)
        topk_vals, topk_idx = torch.topk(z_pre, self.top_k, dim=-1)
        z = torch.zeros_like(z_pre)
        z.scatter_(-1, topk_idx, torch.relu(topk_vals))
        return self.decoder(z), z

# Load model
sae = SAE(ckpt['d_in'], ckpt['d_hidden'], ckpt.get('top_k', 64)).to(DEVICE)
sae.load_state_dict(ckpt['sae_state_dict'])
sae.eval()
mu, sigma = ckpt['mu'].to(DEVICE), ckpt['sigma'].to(DEVICE)
print("SAE loaded!")


SAE loaded!


In [5]:
# Load and filter data
acts_raw = np.load(DATA_PATH, mmap_mode='r')
with open(METADATA_PATH) as f:
    metadata_raw = json.load(f)

# Filter out broken clips (low variance)
clip_vars = acts_raw.mean(axis=1).var(axis=1)
good_idx = np.where(clip_vars > 0.001)[0]
acts = acts_raw[good_idx]
metadata = [metadata_raw[i] for i in good_idx]
print(f"Using {len(good_idx)} good clips (filtered {len(metadata_raw) - len(good_idx)} broken)")

# Prepare normalized data
X = torch.from_numpy(acts.mean(axis=1).astype(np.float32)).to(DEVICE)
Xn = (X - mu) / sigma
print(f"Data shape: {Xn.shape}")


Using 344 good clips (filtered 656 broken)
Data shape: torch.Size([344, 2048])


In [7]:
with torch.no_grad():
    X_hat, Z = sae(Xn)

# Reconstruction
mse = ((X_hat - Xn) ** 2).mean().item()
var_exp = 1 - mse / Xn.var().item()
print(f"MSE: {mse:.6f}")
print(f"Variance explained: {var_exp:.2%}")

# Sparsity
Z_cpu = Z.cpu().numpy()
active_per_sample = (Z_cpu > 0).sum(axis=1)
sparsity = 1 - active_per_sample.mean() / Z_cpu.shape[1]
print(f"\nMean active features: {active_per_sample.mean():.1f} / {Z_cpu.shape[1]}")
print(f"Sparsity: {sparsity:.2%}")

# Feature frequency
feature_freq = (Z_cpu > 0).mean(axis=0)
dead = (feature_freq == 0).sum()
print(f"Dead features: {dead} ({dead/len(feature_freq):.1%})")


MSE: 0.018766
Variance explained: -18144.94%

Mean active features: 64.0 / 8192
Sparsity: 99.22%
Dead features: 8125 (99.2%)


In [8]:
# Top activated features
active_feats = np.where(feature_freq > 0)[0]
sorted_by_freq = active_feats[np.argsort(feature_freq[active_feats])[::-1]]

print("Top 10 most frequent features:")
for f in sorted_by_freq[:10]:
    print(f"  Feature {f}: {feature_freq[f]:.1%}")


Top 10 most frequent features:
  Feature 74: 100.0%
  Feature 7922: 100.0%
  Feature 7869: 100.0%
  Feature 1422: 100.0%
  Feature 1374: 100.0%
  Feature 1341: 100.0%
  Feature 1258: 100.0%
  Feature 1252: 100.0%
  Feature 1146: 100.0%
  Feature 1140: 100.0%


In [9]:
def get_top_clips(feature_idx, k=20):
    """Get top-k clips for a feature."""
    acts = Z_cpu[:, feature_idx]
    top_idx = np.argsort(acts)[::-1][:k]
    return [(i, acts[i], metadata[i]) for i in top_idx]

def analyze_feature_keys(feature_idx, k=20):
    """Check if feature correlates with musical key."""
    clips = get_top_clips(feature_idx, k)
    keys = [c[2].get('label_key') for c in clips]
    counts = Counter(keys)
    print(f"Feature {feature_idx} - top {k} clips:")
    for key, n in counts.most_common(5):
        print(f"  {key}: {n} ({n/k:.0%})")

# Analyze top features
for f in sorted_by_freq[:5]:
    analyze_feature_keys(f)
    print()


Feature 74 - top 20 clips:
  D#_major: 4 (20%)
  G_minor: 2 (10%)
  C_major: 2 (10%)
  G#_major: 2 (10%)
  G_major: 1 (5%)

Feature 7922 - top 20 clips:
  E_minor: 3 (15%)
  G_major: 2 (10%)
  G#_minor: 2 (10%)
  C_minor: 2 (10%)
  A_minor: 2 (10%)

Feature 7869 - top 20 clips:
  C_minor: 5 (25%)
  G_major: 3 (15%)
  D_major: 2 (10%)
  G_minor: 2 (10%)
  A#_major: 2 (10%)

Feature 1422 - top 20 clips:
  C_minor: 4 (20%)
  D#_major: 2 (10%)
  G_major: 2 (10%)
  C_major: 2 (10%)
  C#_major: 2 (10%)

Feature 1374 - top 20 clips:
  G#_minor: 3 (15%)
  E_minor: 3 (15%)
  A#_major: 2 (10%)
  D#_major: 2 (10%)
  D_major: 2 (10%)



## Visualizations


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Feature activation frequency (excluding dead)
axes[0].hist(feature_freq[feature_freq > 0], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Activation Frequency')
axes[0].set_ylabel('Number of Features')
axes[0].set_title('Feature Activation Frequency')

# Per-sample reconstruction error
per_sample_mse = ((X_hat - Xn) ** 2).mean(dim=1).cpu().numpy()
axes[1].hist(per_sample_mse, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Reconstruction MSE')
axes[1].set_ylabel('Count')
axes[1].set_title('Per-Sample Reconstruction Error')

plt.tight_layout()
plt.show()


## Summary


In [ ]:
print("=" * 50)
print("SAE SUMMARY")
print("=" * 50)
print(f"Architecture: {ckpt['d_in']} -> {ckpt['d_hidden']} (TopK={ckpt.get('top_k')})")
print(f"Dataset: {len(metadata)} clips, Layer {LAYER_IDX}")
print(f"MSE: {mse:.6f}, Variance explained: {var_exp:.2%}")
print(f"Sparsity: {sparsity:.2%}, Dead features: {dead}/{len(feature_freq)}")
print("=" * 50)
